# 🇨🇳 Análise dos Empréstimos Chineses
**Datasets utilizados:**
- How China Lends (HCL) v2.0 — cláusulas contratuais detalhadas
- AidData GCDF v2.0 — base principal de projetos
- BRI Tagged v1.0 — classificação BRI por projeto
- How China Collateralizes v1.0 — garantias e colaterais

## 1. Carregamento e União dos Datasets

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

# --- Carregar os 4 datasets ---
hcl  = pd.read_excel('How_China_Lends_Dataset_Version_2_0.xlsx', sheet_name='ContractData')
gcdf = pd.read_excel('AidDatasGlobalChineseDevelopmentFinanceDataset_v2_0.xlsx', sheet_name='Global_CDF2.0')
bri  = pd.read_excel('BRI_Tagged_Chinese_Official_Finance_Dataset_v1.xlsx', sheet_name='Sheet 1')
col  = pd.read_excel('How_China_Collateralizes_Dataset_Version_1_0.xlsx', sheet_name='Data')

# --- Unir GCDF + BRI tags ---
colunas_bri = [
    'AidData TUFF Project ID',
    'BRI_1 Policy Communication', 'BRI_2 Infrastructure Connectivity',
    'BRI_3 Trade', 'BRI_4 Monetary Circulation', 'BRI_5 People to People',
    'BRI-Like', 'Non-BRI', 'Vague-BRI'
]
df = pd.merge(gcdf, bri[colunas_bri], on='AidData TUFF Project ID', how='left')

# --- Unir com Colaterais ---
col_renamed = col.rename(columns={'AidData Record ID': 'AidData TUFF Project ID'})
df = pd.merge(df, col_renamed, on='AidData TUFF Project ID', how='left')

# --- Unir com HCL (clausulas contratuais) ---
hcl_cols = [
    'AidData_record_ID', 'borrower_country', 'borrower_region', 'year',
    'commitment_USD', 'creditor_name', 'project_title',
    'confidentiality_general', 'confidentiality_borrower',
    'paris_clause', 'cross_default', 'penalty_interest_rate',
    'governing_law', 'arbitration', 'collateral',
    'escrow_account', 'reserve_account', 'revenue_account'
]
hcl_slim = hcl[hcl_cols].copy()
hcl_slim['AidData_record_ID'] = hcl_slim['AidData_record_ID'].astype(str)
df['AidData TUFF Project ID'] = df['AidData TUFF Project ID'].astype(str)
df = pd.merge(df, hcl_slim, left_on='AidData TUFF Project ID', right_on='AidData_record_ID', how='left')

# --- Renomear colunas principais ---
df = df.rename(columns={
    'Recipient_x': 'País',
    'Recipient Region_x': 'Região',
    'Commitment Year_x': 'Ano',
    'Sector Name': 'Setor',
    'Amount (Constant USD2017)': 'Valor_USD'
})

print(f'✅ Dataset unido: {df.shape[0]} projetos, {df.shape[1]} colunas')
df[['País', 'Região', 'Ano', 'Setor', 'Valor_USD', 'BRI-Like', 'paris_clause']].head()

---
## 2. Projetos BRI por Região e País

In [ ]:
bri_df = df[df['BRI-Like'] == 1].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

projetos_por_regiao = bri_df.groupby('Região')['AidData TUFF Project ID'].count().sort_values()
projetos_por_regiao.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Número de Projetos BRI por Região')
axes[0].set_xlabel('Quantidade de Projetos')

valor_por_regiao = (bri_df.groupby('Região')['Valor_USD'].sum() / 1e9).sort_values()
valor_por_regiao.plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('Valor Total por Região (Bilhões USD)')
axes[1].set_xlabel('Bilhões de USD (constante 2017)')

plt.suptitle('Projetos BRI por Região', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Top 15 países com mais projetos BRI
top_paises = bri_df.groupby('País')['AidData TUFF Project ID'].count().sort_values(ascending=False).head(15)
top_paises.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 15 Países com Mais Projetos BRI')
plt.xlabel('Quantidade de Projetos')
plt.tight_layout()
plt.show()

---
## 3. Setores Mais Financiados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

setores_valor = (df.groupby('Setor')['Valor_USD'].sum().dropna().sort_values(ascending=False).head(10) / 1e9)
setores_valor.sort_values().plot(kind='barh', ax=axes[0], color='seagreen')
axes[0].set_title('Top 10 Setores por Valor (Bilhões USD)')
axes[0].set_xlabel('Bilhões de USD (constante 2017)')

setores_count = df['Setor'].value_counts().head(10).sort_values()
setores_count.plot(kind='barh', ax=axes[1], color='mediumpurple')
axes[1].set_title('Top 10 Setores por Nº de Projetos')
axes[1].set_xlabel('Quantidade de Projetos')

plt.suptitle('Setores Mais Financiados pela China', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# BRI vs Não-BRI por setor
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bri_df['Setor'].value_counts().head(8).sort_values().plot(kind='barh', ax=axes[0], color='darkorange')
axes[0].set_title('Top Setores — Projetos BRI')

df[df['Non-BRI'] == 1]['Setor'].value_counts().head(8).sort_values().plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top Setores — Projetos Não-BRI')

plt.suptitle('Setores: BRI vs Não-BRI', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Colaterais e Garantias

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

total_com = (df['Collateralized/Securitized'] == 'Yes').sum()
total_sem = df.shape[0] - total_com
axes[0].pie(
    [total_com, total_sem],
    labels=['Com Colateral', 'Sem Colateral'],
    autopct='%1.1f%%',
    colors=['tomato', 'lightgray'],
    startangle=90
)
axes[0].set_title('Proporção de Projetos com Colateral')

commodities = df['Commodity'].value_counts().dropna().head(8).sort_values()
if len(commodities) > 0:
    commodities.plot(kind='barh', ax=axes[1], color='tomato')
    axes[1].set_title('Commodities Usadas como Colateral')
    axes[1].set_xlabel('Quantidade de Projetos')
else:
    axes[1].text(0.5, 0.5, 'Dados insuficientes', ha='center', va='center')
    axes[1].set_title('Commodities Usadas como Colateral')

plt.suptitle('Colaterais e Garantias', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Países com mais empréstimos colateralizados
col_paises = df[df['Collateralized/Securitized'] == 'Yes'].groupby('País')['AidData TUFF Project ID'].count().sort_values(ascending=False).head(12)
col_paises.sort_values().plot(kind='barh', color='tomato')
plt.title('Países com Mais Empréstimos Colateralizados')
plt.xlabel('Quantidade de Projetos')
plt.tight_layout()
plt.show()

---
## 5. Evolução Temporal dos Empréstimos

In [ ]:
(df.groupby('Ano')['Valor_USD'].sum() / 1e9).plot(kind='line', marker='o', color='steelblue', linewidth=2)
plt.title('Evolução dos Empréstimos Chineses ao Longo do Tempo')
plt.ylabel('Bilhões de USD (constante 2017)')
plt.xlabel('Ano')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# BRI vs Não-BRI ao longo do tempo
evolucao_bri    = df[df['BRI-Like'] == 1].groupby('Ano')['Valor_USD'].sum() / 1e9
evolucao_naobri = df[df['Non-BRI'] == 1].groupby('Ano')['Valor_USD'].sum() / 1e9

plt.figure(figsize=(12, 5))
evolucao_bri.plot(label='BRI', marker='o', color='darkorange', linewidth=2)
evolucao_naobri.plot(label='Não-BRI', marker='s', color='steelblue', linewidth=2)
plt.title('Evolução Temporal: BRI vs Não-BRI')
plt.ylabel('Bilhões de USD (constante 2017)')
plt.xlabel('Ano')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Por região ao longo do tempo
evolucao_regiao = (df.groupby(['Ano', 'Região'])['Valor_USD'].sum().unstack().fillna(0) / 1e9)
evolucao_regiao.plot(kind='area', stacked=True, colormap='tab10', alpha=0.8)
plt.title('Empréstimos por Região ao Longo do Tempo')
plt.ylabel('Bilhões de USD (constante 2017)')
plt.xlabel('Ano')
plt.legend(loc='upper left', fontsize=9)
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

---
## 6. Cláusulas Contratuais (exclusivo do HCL)
> Analisa as condições legais presentes nos contratos de empréstimo chineses

In [ ]:
clausulas = {
    'Confidencialidade (Geral)': 'confidentiality_general',
    'Confidencialidade (Tomador)': 'confidentiality_borrower',
    'Cláusula Paris': 'paris_clause',
    'Cross-Default': 'cross_default',
    'Juros de Penalidade': 'penalty_interest_rate',
    'Conta Escrow': 'escrow_account',
    'Conta Reserva': 'reserve_account',
    'Conta Receita': 'revenue_account',
    'Colateral': 'collateral',
    'Arbitragem': 'arbitration',
}

presenca = {}
for label, col_name in clausulas.items():
    if col_name in hcl.columns:
        presenca[label] = (hcl[col_name] == 1).sum()

presenca_series = pd.Series(presenca).sort_values()
total = len(hcl)

ax = presenca_series.plot(kind='barh', color='steelblue', figsize=(10, 6))
for i, v in enumerate(presenca_series):
    ax.text(v + 1, i, f'{v/total*100:.0f}%', va='center', fontsize=9)

plt.title(f'Presença de Cláusulas nos Contratos Chineses (n={total})')
plt.xlabel('Número de Contratos')
plt.xlim(0, presenca_series.max() * 1.15)
plt.tight_layout()
plt.show()

In [ ]:
# Lei governante dos contratos
gov_law = hcl['governing_law'].value_counts().head(10).sort_values()
gov_law.plot(kind='barh', color='mediumpurple')
plt.title('Lei Governante dos Contratos Chineses')
plt.xlabel('Número de Contratos')
plt.tight_layout()
plt.show()

In [ ]:
# Evolução do uso de cláusulas ao longo do tempo
clausulas_tempo = hcl.groupby('year')[['paris_clause', 'cross_default', 'collateral', 'confidentiality_general']].sum()
clausulas_tempo = clausulas_tempo.rename(columns={
    'paris_clause': 'Paris Club',
    'cross_default': 'Cross-Default',
    'collateral': 'Colateral',
    'confidentiality_general': 'Confidencialidade'
})

clausulas_tempo.plot(marker='o', linewidth=2)
plt.title('Evolução do Uso de Cláusulas nos Contratos ao Longo do Tempo')
plt.ylabel('Número de Contratos')
plt.xlabel('Ano')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()